In [3]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path().resolve().parents[0]))

In [6]:
from base_data_pipeline import get_processed_data
from base_data_pipeline import load_dataset

In [5]:
train_texts, test_texts, train_labels, test_labels = get_processed_data()

In [7]:
df = load_dataset()

In [8]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.utils.data import Dataset, DataLoader

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

In [11]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [12]:
class SMSDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [13]:
train_dataset = SMSDataset(train_encodings, train_labels)
test_dataset = SMSDataset(test_encodings, test_labels)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(torch.cuda.is_available)
print(device)

<function is_available at 0x0000016E1CF011C0>
cpu


In [17]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=5e-5)

In [16]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import copy

num_epochs = 10
patience = 2

best_f1 = 0
epochs_no_improve = 0
best_model_state = None

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        loss = outputs.loss
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Training Loss: {avg_loss}")

    # evaluation
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in test_loader:

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(batch["labels"].cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average="binary"
    )

    print("Accuracy:", accuracy)
    print("F1 Score:", f1)

    if f1 > best_f1:

        best_f1 = f1
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0

        print("Improvement detected. Saving model.")

    else:

        epochs_no_improve += 1
        print("No improvement. Patience counter:", epochs_no_improve)

    if epochs_no_improve >= patience:

        print("\nEarly stopping triggered.")
        break


model.load_state_dict(best_model_state)

print("\nBest model restored with F1:", best_f1)

100%|██████████| 279/279 [01:33<00:00,  2.97it/s]



Epoch 1 Training Loss: 0.0889900251392669
Accuracy: 0.9228699551569507
F1 Score: 0.774869109947644
Improvement detected. Saving model.


100%|██████████| 279/279 [01:36<00:00,  2.88it/s]



Epoch 2 Training Loss: 0.06792615159806098
Accuracy: 0.989237668161435
F1 Score: 0.9586206896551724
Improvement detected. Saving model.


100%|██████████| 279/279 [01:39<00:00,  2.80it/s]


...


Epoch 4 Training Loss: 0.019526834473244204
Accuracy: 0.9937219730941704
F1 Score: 0.9764309764309764
Improvement detected. Saving model.


100%|██████████| 279/279 [01:40<00:00,  2.77it/s]



Epoch 5 Training Loss: 0.03386434936840888
Accuracy: 0.9865470852017937
F1 Score: 0.9473684210526315
No improvement. Patience counter: 1


100%|██████████| 279/279 [01:40<00:00,  2.77it/s]



Epoch 6 Training Loss: 0.12233689128785097
Accuracy: 0.9847533632286996
F1 Score: 0.9415807560137457
No improvement. Patience counter: 2

Early stopping triggered.

Best model restored with F1: 0.9764309764309764


In [17]:
!cp /content/drive/MyDrive/Colab\ Notebooks/evaluation_utils.py /content

In [18]:
import evaluation_utils

In [19]:
results = evaluation_utils.evaluate_model(true_labels, predictions)

In [20]:
results

{'Accuracy': 0.9847533632286996,
 'Precision': 0.9647887323943662,
 'Recall': 0.9194630872483222,
 'F1 Score': 0.9415807560137457,
 'Confusion Matrix': array([[961,   5],
        [ 12, 137]])}